# Inicialização

In [28]:
%pip install selenium pandas

import os
import platform
import shutil
import socket
import subprocess
import time
import ctypes

import re
import json
from datetime import date

from pathlib import Path
from ctypes import wintypes

from selenium import webdriver
from selenium.webdriver.chrome.options import Options as ChromeOptions
from selenium.webdriver.edge.options import Options as EdgeOptions
from selenium.webdriver.common.by import By
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys

port = 9222
profile = Path.home() / ".sefaz_automacao_perfil"
url = 'https://plataforma.sefaz.ma.gov.br/'

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [ ]:
def fdoc():
    if platform.system() == 'Windows':
        try:
            buf = ctypes.create_unicode_buffer(wintypes.MAX_PATH)
            if ctypes.windll.shell32.SHGetFolderPathH(None, 5,
                                                      None, 0,
                                                      buf) == 0:
                return Path(buf.value)
        except:
            pass
    else:
        cfg = Path.home() / '.config' / 'user-dirs.dirs'
        if cfg.exists():
            for line in cfg.read_text(encoding = 'utf-8').splitlines():
                if line.startswith('XDG_DOCUMENTS_DIR='):
                    path = line.split('=', 1)[1].strip().strip('"')
                    return Path(path.replace('$HOME', str(Path.home())))
    return Path.home() / 'Documetos'

folder = fdoc() / 'Sefaz MA'
cache = folder / 'cache' / 'datas consultadas'
cache.parent.mkdir(parents = True, exist_ok = True)

interval = 60 * 60

def lcache():
    if cache.exists():
        try:
            return json.load(cache.read_text(encoding = 'utf-8'))
        except json.JSONDecodeError:
            return {}
    return {}

def scache(date):
    tmp = cache.with_suffix('.tmp')
    tmp.write_text(json.dumps(date, ensure_ascii = False,
                              indent = 2), encoding = 'utf-8')
    tmp.replace(cache)

def key(name, b, e):
    cnpj = re.search(r'CNPJ: [\d./-]+', name).group(1)
    return f'{cnpj}|{b}|{e}'

def need_consulting(k):
    last = lcache().get(k)
    return last is None or (time.time() - last) >= interval

def write_con(k):
    dates = lcache()
    dates[k] = time.time()
    scache(dates)

## funções

In [16]:
def _cbrowser():
    sis = platform.system()
    cand = []

    if sis == 'Windows':
        bases = [os.environ.get(v) for v in
                 ('PROGRAMFILES', 'PROGRAMFILES(X86)', 'LOCALAPPDATA')]
        for base in filter(None, bases):
            cand.append(('chrome', Path(base) / 'Google/Chrome/Application/chrome.exe'))
        for base in filter(None, bases):
                    cand.append(('edge', Path(base) / 'Microsoft/Edge/Application/msedge.exe'))
    else:
          for nome in ('google-chrome', 'google-chrome-stable', 'chromium', 'chromium-browser'):
                if (p := shutil.which(nome)):
                      cand.append(('chrome', Path(p)))
    return cand

In [17]:
def _fbrowser():
    for type, path in _cbrowser():
        if path.exists():
            return type, str(path)
    raise RuntimeError('Nenhum navegador compativel... Por favor, instale Edge ou Chrome')

In [18]:
def _openport(port):
    with socket.socket() as s:
        s.settimeout(0.5)
        return s.connect_ex(('127.0.0.1', port)) == 0

In [19]:
def _tab(drv, ad = url):
    for h in drv.window_handles:
        drv.switch_to.window(h)
        if ad in drv.current_url:
            return
    drv.switch_to.window(drv.windwo_handles[-1])

In [20]:
def obrowser():
    type, exe = _fbrowser()
    profile.mkdir(parents = True, exist_ok = True)

    if not _openport(port):
        subprocess.Popen([
            exe,
            f'--remote-debugging-port={port}',
            f'--user-data-dir={profile}',
            '--no-first-run',
            '--no-default-browser-check',
            url
        ])
        for _ in range(30):
            if _openport(port):
                break
            time.sleep(0.5)
        else:
            raise RuntimeError('O navegador não abriu a porta a tempo')

    input(
        '\nO navegador foi aberto. Faça o login e acesse a plataforma.\n'
        'Após realizado o login, retorne à esta página e aperte ENTER para continuar'
    )

    if type == 'edge':
        opt = EdgeOptions()
        opt.add_experimental_option('debuggerAddress', f'127.0.0.1:{port}')
        drv = webdriver.Edge(options = opt)
    else:
        opt = ChromeOptions()
        opt.add_experimental_option('debuggerAddress', f'127.0.0.1:{port}')
        drv = webdriver.Chrome(options = opt)

    _tab(drv)
    return drv


# Automatização

In [23]:
browser = obrowser()
wait = WebDriverWait(browser, timeout = 15)
print('Conectado via:', browser.current_url)




DevTools listening on ws://127.0.0.1:9222/devtools/browser/2922d702-1532-4394-9890-8b701b936a3d


Conectado via: https://plataforma.sefaz.ma.gov.br/#/


In [26]:
btt = (By.XPATH, '/html/body/app-root/app-main/div/app-topbar/div/a/i')

wait.until(EC.element_to_be_clickable(btt)).click()

btt = (By.XPATH, '/html/body/app-root/app-main/div/div[1]/app-menu/div/ul/li/ul/li[2]/a/span')

wait.until(EC.element_to_be_clickable(btt)).click()

ddfe = wait.until(EC.element_to_be_clickable((By.XPATH, '/html/body/app-root/app-main/div/div[1]/app-menu/div/ul/li/ul/li[2]/ul/li[1]/a/span')))

ddfe.click()

In [ ]:
selem = (By.XPATH, '/html/body/app-root/app-main/div/div[2]/div/app-download-form/p-card[1]/div/div/div[2]/form/div[2]/div/p-dropdown/div/span')

xpath_opt = "//li[contains(@class,'p-dropdown-item')]"

def norm(t):
    return re.sub(r'\s+', ' ', t).strip()

def olist():
    wait.until(EC.element_to_be_clickable(selem)).click()
    wait.until(EC.presence_of_all_elements_located((By.XPATH, xpath_opt)))

def listlist():
    olist()
    itens = browser.find_elements(By.XPATH, xpath_opt)
    names = [norm(i.get_attribute('aria-label') or i.text) for i in itens]
    browser.find_element(By.TAG_NAME, 'body').send_keys(Keys.ESCAPE)
    return [n for n in names if n]

def select(name):
    olist()
    itens = browser.find_elements(By.XPATH, xpath_opt)
    for item in itens:
        if norm(item.get_attribute('aria-label') or item.text) == name:
            browser.execute_script("arguments[0].scrollIntoView({block: 'center'});",
                                  item)
            item.click()
            return
    raise ValueError(f'Empresa não encontrada: {name}')

firms = listlist()
print(f'{len(firms)} empresas encontradas')




156 empresas encontradas


"for i, name in enumerate(firms, 1):\n    try:\n        select(name)\n    except Exception as e:\n        print(f'Erro - Selecionar {name}: {e}')"

In [ ]:

now = date.today()
b = now.replace(day = 1)
e = now

b = b.isoformat()
e = e.isoformat()

for i, name in enumerate(firms, 1):
    k = key(name, b, e)

    if not need_consulting(k):
        print(f'[{i}/{len(firms)}] {name}: consultado há menos de 1h. Pulando...')

    m = str(now.month)
    y = str(now.year)

    try:
        select(name)

        square = (By.XPATH, '//*[@id="competencia"]')

        data = wait.until(EC.element_to_be_clickable(square)).click()
        data.send_keys(Keys.CONTROL, 'a')
        data.send_keys(Keys.DELETE)
        data.send_keys(m, y)

        tipos = ('NFE', 'NFCE')

        for tipo in tipos:
            print(f'Iniciando o download das {tipo}...')

            wait.until(
                EC.element_to_be_clickable(
                    (By.XPATH,
                     '/html/body/app-root/app-main/div/div[2]/div/app-download-form/p-card[1]/div/div/div[2]/form/div[3]/div[3]/div/div[2]/p-dropdown/div/span')
                     )).click()
            
            opcao = wait.until(EC.element_to_be_clickable((By.XPATH, f"//li[contains(@class,'p-dropdown-item')][@aria-label='{tipo}']")))

            wait.until(EC.element_to_be_clickable((By.XPATH, '//*[@id="pr_id_4_label"]'))).click()

            wait.until(EC.element_to_be_clickable((By.XPATH, '///*[@id="p-highlighted-option"]'))).click()

            wait.until(EC.element_to_be_clickable((By.XPATH, '/html/body/app-root/app-main/div/div[2]/div/app-download-form/p-card[1]/div/div/div[2]/form/div[3]/div[7]/div/div[2]/div/button')))

    except Exception as e:
        print(f'Erro - Selecionar {name}: {e}')